#4 - Limpeza na Camada Silver - Micro-Batch

---
Importando todas as bibliotecas que serão utilizadas durante o notebook, em seguida definimos todas as variáveis que serão utilizadas durante a execução.

#.
### Variáveis e suas utilizações
micro_batch_bronze_path = define o caminho de leitura da tabela bronze do micro batch

micro_batch_silver_path = define o caminho onde será salva a tabela silver do micro batch

---

In [0]:
import pyspark.sql.functions as sf
from pyspark.sql.window import Window

micro_batch_bronze_path = "workspace.stocks.micro_batch_bronze"
micro_batch_silver_path = "workspace.stocks.micro_batch_silver"

---

Verificamos o último timestamp salvo na tabela silver micro batch para garantir que somente registros novos sejam processados a cada execução, evitando duplicidade de dados. Caso a tabela ainda não exista, retornamos None e processamos tudo.

---

In [0]:
try:
      ultimo_ts = (
            spark.read.table(micro_batch_silver_path)
            .agg(sf.max("event_time"))
            .collect()[0][0]
      )
      print(f"-Ultimo TimeStamp: {ultimo_ts}-")
except:
      ultimo_ts = None

---

Lemos a tabela bronze do micro batch e fazemos os CASTs de tipos para padronização do valores passados para cada coluna.

---

In [0]:
df = spark.read.table(micro_batch_bronze_path)

df = (df.withColumn("event_time", sf.to_timestamp(sf.col("datetime")))
           .withColumn("ingestao_ts", sf.current_timestamp())
           .withColumn("open", sf.col("open").cast("double"))
           .withColumn("high", sf.col("high").cast("double"))
           .withColumn("low", sf.col("low").cast("double"))
           .withColumn("close", sf.col("close").cast("double"))
           .withColumn("volume", sf.col("volume").cast("long"))
           .drop("datetime")
        ).select(
                "ticker", "event_time","open", "high", "low", "close", "volume","fonte", "ingestao_ts"
        )
print("-----Feita Cast de Tipos-----")

---

Fazemos a deduplicação dos dados que podem ser gerados na tabela bronze durante a execulção e também retiramos valores nulos da coluna, para manter somente linhas completas e com qualidade de dados.

---

In [0]:
df = df.dropDuplicates(["ticker", "event_time"])
df = df.dropna()
print("-----Removidos Duplicatas e Nulos-----")

---

Fazemos o enriquecimento da tabela, adicionando colunas novas que não vem calculadas da base da API, essas colunas serão utilizadas para acompanhar com maior precisão a variação temporal de cada ação.

Também retiramos valores invalidos das colunas, evitando imprevistos de valores mal colocados ou carregados de forma incorreta.

---

In [0]:
df = (df.withColumn("week_year", sf.concat(sf.weekofyear("event_time"), sf.lit("-"), sf.year("event_time")))
      .withColumn("variacao_real", (sf.col("close") - sf.col("open")))
      .withColumn("variacao_percent", (sf.col("variacao_real") / sf.col("open")*100))
      .withColumn("flag_valor_invalido",
                   sf.when(
                           (sf.col("open") <= 0) |
                           (sf.col("high") <= 0) |
                           (sf.col("low") <= 0) |
                           (sf.col("close") <= 0) |
                           (sf.col("volume") < 0), 
                           True
                   ).otherwise(False)
                   ))

df = df.filter(sf.col("flag_valor_invalido") == False)
df = df.drop("flag_valor_invalido")
print("-----Tabela Limpa e Enriquecida-----")

---

Filtramos apenas os registros mais recentes que o último timestamp salvo, garantindo que somente candles novos sejam adicionados à tabela silver microbatch.

Salvamos em append com particionamento por ticker para otimizar as consultas do gráfico.

---

In [0]:
if ultimo_ts is not None:
      df = df.filter(sf.col("event_time") > ultimo_ts)

(
    df.write
    .format("delta")
    .mode("append")
    .partitionBy("ticker")
    .saveAsTable(micro_batch_silver_path)
)

---

Verificamos se novos registros foram gravados e exibimos uma amostra dos dados do primeiro ticker ordenados por event_time para confirmar a correta atualização da tabela.

---

In [0]:
if df.count() > 0:
      print("-Dados gravados-")
      display(df
      .filter(sf.col("ticker") == df.first()["ticker"])
      .orderBy(sf.col("event_time"))
      )
else:
    print("-Sem valores novos-")